# 3 · Aggregation — a number instead of a subgraph

`graph.aggregate()` takes the same `Start` and `Hop`s as `traverse()` and returns
numbers, computed **in the database**, in one round trip — none of the edge
reconstruction or hydration a traversal pays for.

The one rule to carry: the aggregates run over the **distinct nodes the last hop
matched** (the seed set when there are no hops), each node counted once however
many paths reach it.

In [1]:
from demo_graph import connect, names, seed
from hopai import Avg, Count, Hop, Max, Min, Start, Sum

graph = connect("nb_03_aggregation")
seed(graph)
print(names(graph.traverse(Start())))

['Acme', 'Alice', 'Bob', 'Carol', 'Dave', 'Erin', 'Globex']


## The five aggregates

Several in one call is one statement, not one per name.

In [2]:
graph.aggregate(
    Start(where={"type": "person"}),
    aggregates={
        "people": Count(),
        "avg_age": Avg("age"),
        "youngest": Min("age"),
        "oldest": Max("age"),
        "total_age": Sum("age"),
    },
)

{'people': 5, 'avg_age': 35.8, 'youngest': 23, 'oldest': 52, 'total_age': 179}

Values come back as plain `int`/`float` — `json.dumps`-clean, because an
aggregation result is exactly the thing that gets serialized straight into a tool
response. (The driver hands back `Decimal` for `NUMERIC`; that is converted here,
and an integral value comes back as `3`, not `3.0`.)

### `Count()` versus `Count("property")`

Bare `Count()` counts the nodes. `Count("city")` counts the nodes **carrying that
property** — Erin has no `city`, so the two disagree, which is the point.

In [3]:
graph.aggregate(
    Start(where={"type": "person"}),
    aggregates={"people": Count(), "with_city": Count("city"), "cities": Count("city", distinct=True)},
)

{'people': 5, 'with_city': 4, 'cities': 2}

`distinct=True` collapses equal property values first: two people in Berlin and two
in Lisbon make four `with_city` and two `cities`. It works on `Count`, `Sum` and
`Avg` — `Sum("age", distinct=True)` adds each distinct age once, which is a real
question sometimes and a bug the rest of the time, so it is opt-in.

## Over a traversal

Add hops and the aggregate follows the last one. "How old is Alice's network, on
average, out to four friend hops?"

In [4]:
graph.aggregate(
    Start(where={"name": "Alice"}),
    Hop(via={"kind": "friend"}, hops=(1, 4)),
    aggregates={"reached": Count(), "avg_age": Avg("age")},
)

{'reached': 4, 'avg_age': 36.25}

Compare that with the traversal it mirrors: the traversal reports Alice too,
because she is an endpoint of the edges it walked. The aggregate does not — she is
not a node the *last hop matched*.

In [5]:
hop = Hop(via={"kind": "friend"}, hops=(1, 4))
walk = graph.traverse(Start(where={"name": "Alice"}), hop)
counted = graph.aggregate(Start(where={"name": "Alice"}), hop, aggregates={"reached": Count()})

print("traverse reports:", names(walk))
print("aggregate counts:", counted["reached"], "-- the nodes the hop landed on, Alice not among them")

traverse reports: ['Alice', 'Bob', 'Carol', 'Dave', 'Erin']
aggregate counts: 4 -- the nodes the hop landed on, Alice not among them


Each node is counted **once**, however many paths reach it. Dave is reachable
through Bob and through Carol; he counts once. That is the whole reason bare
per-path aggregation from Cypher is refused rather than approximated (notebook 04).

## The last hop, and only the last hop

There is no way to aggregate a middle step, and that is deliberate: a mid-chain
match includes nodes with no continuation to the end of the chain, so aggregating
one would count nodes the equivalent Cypher query would not. Aggregate the chain
you actually mean:

In [6]:
# "What is the average founding year of the companies Alice's network works for?"
graph.aggregate(
    Start(where={"name": "Alice"}),
    Hop(via={"kind": "friend"}, hops=(1, 4), where={"active": True}),
    Hop(via={"kind": "works_at"}, where={"type": "company"}),
    aggregates={"companies": Count(), "avg_founded": Avg("founded")},
)

{'companies': 1, 'avg_founded': 1999}

## Empty matches, and values that are not numbers

Nothing matched is not an error. `count` → `0`, `sum` → `0`, `avg`/`min`/`max` →
`None`.

In [7]:
graph.aggregate(
    Start(where={"type": "unicorn"}),
    aggregates={"n": Count(), "sum": Sum("age"), "avg": Avg("age"), "max": Max("age")},
)

{'n': 0, 'sum': 0, 'avg': None, 'max': None}

And a property that is missing, `null`, or simply not a number is **skipped**, the
way both SQL and Cypher skip `NULL` in an aggregate — one bad row does not error
the query. Watch what that costs: adding a person whose age is the *string*
`"forty"` leaves the average alone but changes the count of people.

In [8]:
graph.add_nodes([{"type": "person", "name": "Frank", "age": "forty", "active": True}])

print(graph.aggregate(Start(where={"type": "person"}),
                      aggregates={"people": Count(), "with_age": Count("age"), "avg_age": Avg("age")}))

{'people': 6, 'with_age': 6, 'avg_age': 35.8}


`with_age` counts Frank (the key is there); `avg_age` ignores him (the value is not
a number). Both are defensible and neither is what you meant, which is why the
right fix is upstream: `PropertyType("age", "number")` keeps such a row out of the
table entirely — notebook 05.

## Why not just count the traversal in Python?

Because the aggregate never leaves the database. No edge CTEs are emitted, nothing
is hydrated, and one number crosses the wire instead of a subgraph. On this
seven-node graph the difference is noise; the shape of the difference is not.

In [9]:
import time

start = time.perf_counter()
subgraph = graph.traverse(Start(where={"type": "person"}), Hop(via={"kind": "friend"}, hops=(1, 4)))
traverse_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
counted = graph.aggregate(Start(where={"type": "person"}), Hop(via={"kind": "friend"}, hops=(1, 4)),
                          aggregates={"n": Count()})
aggregate_ms = (time.perf_counter() - start) * 1000

print(f"traverse : {traverse_ms:5.1f} ms  -> {len(subgraph.nodes)} nodes, {len(subgraph.edges)} edges hydrated")
print(f"aggregate: {aggregate_ms:5.1f} ms  -> {counted}")

traverse :   7.8 ms  -> 5 nodes, 6 edges hydrated
aggregate:   4.9 ms  -> {'n': 5}


`benchmarks/` has the version of this measurement that is worth quoting — on a
million-node graph, and including the cases where raw SQL still wins.

## What is not covered

Each of these refuses with a message rather than approximating: grouping
(`RETURN b.city, count(b)`), edge-property aggregates, `stddev` and percentiles,
and lexicographic `min`/`max` on strings. Numeric properties over the last step's
matched nodes is the whole feature.

---

Next: [04 · JSON and Cypher](04_json_and_cypher.ipynb) — the same engine, addressed
by a tool-calling model and by Cypher.